In [1]:
import pandas as pd

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")

In [3]:
#タイムスタンプを時系列特徴量に変換
df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df['hour'] = df['time_stamp'].dt.hour
df['dayofweek'] = df['time_stamp'].dt.dayofweek

In [4]:
#ユーザー毎の人気度を追加
df['user_interest'] = df[df['event_type'].isin([3])].groupby('user_id')['event_type'].transform('count')

In [5]:
#商品毎の人気度を追加
df['product_interest'] = df[df['event_type'].isin([3])].groupby('product_id')['event_type'].transform('count')

In [6]:
# 人気度を基に順位付け（降順）、同じ人気度の場合は product_id を基準に昇順で順位付け
df['product_interest_rank'] = df.sort_values(by=['product_interest', 'product_id'], ascending=[False, True]) \
                                  .groupby('product_id') \
                                  .cumcount() + 1

In [6]:
#event_typeが3以外だとNaNになるから0で埋める
df = df.fillna(0)

In [7]:
#前処理結果をcsvへ
df.to_csv('./train/train_A.csv', index=False)

In [9]:
# product_interest_rank の降順でデータを並べ替え
df_sorted = df.sort_values(by='product_interest_rank', ascending=False)

In [10]:
df_extract = pd.DataFrame()
df_extract = df_sorted.drop(['user_id', 'event_type', 'time_stamp', 'hour', 'dayofweek', 'ad', 'user_interest'], axis=1)

In [11]:
#ソート結果をcsvへ
df_sorted.to_csv('./train/train_sort_A.csv', index=False)

In [12]:
df_extract.to_csv('./train/train_extract_A.csv', index=False)